# Evaluation — SFT vs GRPO on PokerBench Test Set

Runs both adapters on 1000 test examples and compares accuracy.

**Setup:** Kaggle → Settings → Accelerator → GPU T4 x2 → Internet → On

**Data:** Add your SFT and GRPO adapter datasets via Add Data on the right sidebar.

In [1]:
!pip install -q transformers datasets peft bitsandbytes accelerate unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.2/29.2 MB 58.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 38.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594

## 1. Set adapter paths

Change these to match your Kaggle dataset paths.

In [2]:
import os

# ← CHANGE THESE PATHS to your Kaggle dataset locations
SFT_ADAPTER_DIR = "/kaggle/input/models/dominicvdb/pokerapp-sft-adapter/transformers/default/2"
GRPO_ADAPTER_DIR = "/kaggle/input/models/dominicvdb/grpo-checkpoint/transformers/default/1"

# How many test examples to evaluate (max 11,000)
NUM_TEST_EXAMPLES = 500

assert os.path.exists(SFT_ADAPTER_DIR), f"SFT adapter not found at {SFT_ADAPTER_DIR}"
assert os.path.exists(GRPO_ADAPTER_DIR), f"GRPO adapter not found at {GRPO_ADAPTER_DIR}"
print(f"SFT adapter:  {os.listdir(SFT_ADAPTER_DIR)}")
print(f"GRPO adapter: {os.listdir(GRPO_ADAPTER_DIR)}")

SFT adapter:  ['adapter_model.safetensors', 'adapter_config.json', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']
GRPO adapter: ['adapter_model.safetensors', 'trainer_state.json', 'training_args.bin', 'adapter_config.json', 'tokenizer.json', 'tokenizer_config.json', 'scaler.pt', 'chat_template.jinja', 'scheduler.pt', 'optimizer.pt', 'rng_state.pth']


## 2. Reward function and preprocessor

In [3]:
import re

VALID_ACTIONS = ("check", "fold", "call", "bet", "raise")
_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)
_PASSIVE = {"check", "call"}
_AGGRESSIVE = {"bet", "raise"}

SYSTEM_PROMPT = (
    "You are a poker decision engine. Given a game scenario, output only the "
    "optimal action (check, fold, call, bet X, or raise X). Do not explain."
)


def _strip_thinking(text):
    return _THINK_RE.sub("", text).strip()


def parse_action_type(text):
    text = _strip_thinking(text).lower()
    for action in VALID_ACTIONS:
        if text.startswith(action):
            return action
    if "all-in" in text or "allin" in text or "all in" in text:
        return "raise"
    return None


def parse_bet_amount(text):
    text = _strip_thinking(text).lower()
    action = parse_action_type(text)
    if action in ("check", "fold", "call"):
        return None
    match = re.search(r"(\d+(?:\.\d+)?)", text)
    return float(match.group(1)) if match else None


def poker_reward(predicted, correct):
    pred_action = parse_action_type(predicted)
    true_action = parse_action_type(correct)
    if pred_action != true_action:
        both = {pred_action, true_action}
        if both <= _AGGRESSIVE:
            return -0.3
        if both <= _PASSIVE:
            return -0.3
        return -1.0
    true_amount = parse_bet_amount(correct)
    if true_amount is None:
        return 1.0
    if true_amount == 0:
        return 1.0
    pred_amount = parse_bet_amount(predicted)
    if pred_amount is None:
        return 0.1
    ratio = pred_amount / true_amount
    if 0.9 <= ratio <= 1.1:
        return 1.0
    elif 0.8 <= ratio <= 1.2:
        return 0.7
    elif 0.5 <= ratio <= 1.5:
        return 0.4
    else:
        return 0.1


def format_grpo(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["instruction"]},
    ]


def apply_chat_template(messages, tokenizer, add_generation_prompt=False):
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt,
    )


print("Reward function and preprocessor ready")

Reward function and preprocessor ready


## 3. Load test dataset

In [4]:
from datasets import load_dataset

dataset = load_dataset("RZ412/PokerBench", cache_dir="./data")
test_ds = dataset["test"].shuffle(seed=42).select(range(NUM_TEST_EXAMPLES))
print(f"Evaluating on {len(test_ds)} test examples")

README.md: 0.00B [00:00, ?B/s]

postflop_500k_train_set_prompt_and_label(…):   0%|          | 0.00/561M [00:00<?, ?B/s]

preflop_60k_train_set_prompt_and_label.j(…):   0%|          | 0.00/59.2M [00:00<?, ?B/s]

postflop_10k_test_set_prompt_and_label.j(…):   0%|          | 0.00/11.2M [00:00<?, ?B/s]

(…)reflop_1k_test_set_prompt_and_label.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/563200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11000 [00:00<?, ? examples/s]

Evaluating on 500 test examples


## 4. Evaluation function

In [5]:
import torch
from collections import Counter
import time


def evaluate_model(model, tokenizer, test_data, label="Model"):
    """Run evaluation and return detailed results."""
    results = []
    action_correct = Counter()
    action_total = Counter()
    reward_sum = 0.0
    exact_match = 0
    action_match = 0

    t0 = time.time()
    for i, row in enumerate(test_data):
        prompt = apply_chat_template(
            format_grpo(row), tokenizer, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip()
        generated_clean = _strip_thinking(generated)

        reward = poker_reward(generated_clean, row["output"])
        pred_action = parse_action_type(generated_clean)
        true_action = parse_action_type(row["output"])

        reward_sum += reward
        if reward == 1.0:
            exact_match += 1
        if pred_action == true_action:
            action_match += 1
            action_correct[true_action] += 1
        action_total[true_action] += 1

        results.append({
            "expected": row["output"],
            "predicted": generated_clean,
            "reward": reward,
            "pred_action": pred_action,
            "true_action": true_action,
        })

        if (i + 1) % 100 == 0:
            elapsed = time.time() - t0
            print(f"  [{label}] {i+1}/{len(test_data)} — "
                  f"action acc: {action_match/(i+1):.1%}, "
                  f"exact: {exact_match/(i+1):.1%}, "
                  f"avg reward: {reward_sum/(i+1):.3f}, "
                  f"elapsed: {elapsed:.0f}s")

    total = len(test_data)
    elapsed = time.time() - t0

    print(f"\n{'='*60}")
    print(f"  {label} RESULTS ({total} examples, {elapsed:.0f}s)")
    print(f"{'='*60}")
    print(f"  Action type accuracy : {action_match/total:.1%} ({action_match}/{total})")
    print(f"  Exact match (reward=1): {exact_match/total:.1%} ({exact_match}/{total})")
    print(f"  Average reward        : {reward_sum/total:.3f}")
    print(f"\n  Per-action breakdown:")
    for action in sorted(action_total.keys()):
        acc = action_correct[action] / action_total[action] if action_total[action] > 0 else 0
        print(f"    {action:<8}: {action_correct[action]:>4}/{action_total[action]:<4} ({acc:.1%})")
    print()

    return {
        "action_accuracy": action_match / total,
        "exact_match": exact_match / total,
        "avg_reward": reward_sum / total,
        "per_action": {a: action_correct[a]/action_total[a] if action_total[a] > 0 else 0 for a in action_total},
        "results": results,
    }


print("Evaluation function ready")

Evaluation function ready


## 5. Evaluate SFT model

In [6]:
from unsloth import FastLanguageModel

print("Loading SFT model...")
sft_model, sft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_DIR,
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(sft_model)
print(f"SFT model loaded — VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

sft_results = evaluate_model(sft_model, sft_tokenizer, test_ds, label="SFT")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading SFT model...
==((====))==  Unsloth 2026.3.5: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.5 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


SFT model loaded — VRAM: 7.9 GB


--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

  [SFT] 100/500 — action acc: 90.0%, exact: 90.0%, avg reward: 0.800, elapsed: 100s
  [SFT] 200/500 — action acc: 87.5%, exact: 87.5%, avg reward: 0.750, elapsed: 200s
  [SFT] 300/500 — action acc: 88.3%, exact: 88.0%, avg reward: 0.765, elapsed: 302s
  [SFT] 400/500 — action acc: 89.2%, exact: 89.0%, avg reward: 0.783, elapsed: 405s
  [SFT] 500/500 — action acc: 89.4%, exact: 89.2%, avg reward: 0.787, elapsed: 509s

  SFT RESULTS (500 examples, 509s)
  Action type accuracy : 89.4% (447/500)
  Exact match (reward=1): 89.2% (446/500)
  Average reward        : 0.787

  Per-action breakdown:
    bet     :   28/40   (70.0%)
    call    :  105/118  (89.0%)
    check   :  119/127  (93.7%)
    fold    :  115/121  (95.0%)
    raise   :   80/94   (85.1%)



## 6. Free memory, then evaluate GRPO model

In [7]:
import json
with open(f"{GRPO_ADAPTER_DIR}/adapter_config.json") as f:
    config = json.load(f)
print(json.dumps(config, indent=2))

{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": {
    "base_model_class": "Qwen3ForCausalLM",
    "parent_library": "transformers.models.qwen3.modeling_qwen3",
    "unsloth_fixed": true
  },
  "base_model_name_or_path": "unsloth/qwen3-8b-unsloth-bnb-4bit",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 64,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "peft_type": "LORA",
  "peft_version": "0.18.1",
  "qalora_group_size": 16,
  "r": 32,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "gate_proj",
    "up_proj",
    "k_proj",
    "v_proj",
    "q_

In [8]:
# Free SFT model from GPU
import gc
del sft_model
del sft_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

print("\nLoading GRPO model...")
grpo_model, grpo_tokenizer = FastLanguageModel.from_pretrained(
    model_name=GRPO_ADAPTER_DIR,
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(grpo_model)
print(f"GRPO model loaded — VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

grpo_results = evaluate_model(grpo_model, grpo_tokenizer, test_ds, label="GRPO")

VRAM after cleanup: 0.0 GB

Loading GRPO model...
==((====))==  Unsloth 2026.3.5: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
GRPO model loaded — VRAM: 7.9 GB
  [GRPO] 100/500 — action acc: 82.0%, exact: 81.0%, avg reward: 0.631, elapsed: 96s
  [GRPO] 200/500 — action acc: 81.0%, exact: 80.5%, avg reward: 0.615, elapsed: 193s
  [GRPO] 300/500 — action acc: 82.7%, exact: 81.7%, avg reward: 0.646, elapsed: 292s
  [GRPO] 400/500 — action acc: 84.0%, exact: 83.0%, avg reward: 0.673, elapsed: 392s
  [GRPO] 500/500 — action acc: 84.0%, exact: 83.2%, avg reward: 0.675, elapsed: 492s

  GRPO RESULTS (500 examples, 492s)
  Action type accuracy : 84.0% (420/500)
  Exact match (reward=1): 83.2% (416/500)
  Average reward        : 0.675

  Per-action breakdown:
    bet     :   26/40   (65.0%)
    call    :  106/118  (89.8%)
    check   :  117/127  (92.1%)
    fold    :  111/121  (91.7%)
    raise   :   60/94   (63.8%)



In [9]:
# ── Evaluate BASE Qwen3-8B (no adapters) ──

import gc
del grpo_model
del grpo_tokenizer
gc.collect()
torch.cuda.empty_cache()

print("Loading base Qwen3-8B (no fine-tuning)...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/qwen3-8b-unsloth-bnb-4bit",
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(base_model)
print(f"Base model loaded — VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

base_results = evaluate_model(base_model, base_tokenizer, test_ds, label="BASE")

Loading base Qwen3-8B (no fine-tuning)...
==((====))==  Unsloth 2026.3.5: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Base model loaded — VRAM: 7.5 GB
  [BASE] 100/500 — action acc: 0.0%, exact: 0.0%, avg reward: -1.000, elapsed: 256s
  [BASE] 200/500 — action acc: 0.0%, exact: 0.0%, avg reward: -1.000, elapsed: 508s
  [BASE] 300/500 — action acc: 0.0%, exact: 0.0%, avg reward: -1.000, elapsed: 758s
  [BASE] 400/500 — action acc: 0.0%, exact: 0.0%, avg reward: -1.000, elapsed: 1008s
  [BASE] 500/500 — action acc: 0.0%, exact: 0.0%, avg reward: -1.000, elapsed: 1255s

  BASE RESULTS (500 examples, 1255s)
  Action type accuracy : 0.0% (0/500)
  Exact match (reward=1): 0.0% (0/500)
  Average reward        : -1.000

  Per-action breakdown:
    bet     :    0/40   (0.0%)
    call    :    0/118  (0.0%)
    check   :    0/127  (0.0%)
    fold    :    0/121  (0.0%)
    raise   :    0/94   (0.0%)



In [10]:
for r in base_results["results"][:10]:
    print("PRED:", repr(r["predicted"]))
    print("TRUE:", repr(r["expected"]))
    print("PARSED:", r["pred_action"], r["true_action"])
    print()

PRED: "<think>\nOkay, let's break this down. I'm in the big blind with two of hearts and clubs. The HJ raised pre-flop, and"
TRUE: 'call'
PARSED: None call

PRED: "<think>\nOkay, let's break this down. I'm the big blind with K♦2♦. The board is A♦, 9♥,"
TRUE: 'fold'
PARSED: None fold

PRED: "<think>\nOkay, let's break this down. I'm in UTG with A♠ K♥. The action started with UTG raising 2 chips pre"
TRUE: 'bet 24'
PARSED: None bet

PRED: "<think>\nOkay, let's break this down. I'm the button with QhJs. The board is Qc, 8c, Ac, then"
TRUE: 'fold'
PARSED: None fold

PRED: "<think>\nOkay, let's break this down. I'm the big blind with 8♦7♠. The hand started with SB raising to 3,"
TRUE: 'check'
PARSED: None check

PRED: "<think>\nOkay, let's break this down. I'm in the small blind with two nines. The big blind called my raise. The flop was"
TRUE: 'call'
PARSED: None call

PRED: "<think>\nOkay, let's break this down. I'm in HJ with A♣J♦. The board is 5♣, 2♦,"
TRUE: 'raise 41'
PARSED: None rais

## 7. Compare results

In [11]:
print("\n" + "=" * 70)
print("  COMPARISON: BASE vs SFT vs GRPO")
print("=" * 70)
print(f"{'Metric':<25} {'BASE':>10} {'SFT':>10} {'GRPO':>10}")
print("-" * 55)

for metric, label in [
    ("action_accuracy", "Action accuracy"),
    ("exact_match", "Exact match"),
    ("avg_reward", "Avg reward"),
]:
    base_val = base_results[metric]
    sft_val = sft_results[metric]
    grpo_val = grpo_results[metric]
    if metric == "avg_reward":
        print(f"{label:<25} {base_val:>10.3f} {sft_val:>10.3f} {grpo_val:>10.3f}")
    else:
        print(f"{label:<25} {base_val:>10.1%} {sft_val:>10.1%} {grpo_val:>10.1%}")

print(f"\n{'Per-action accuracy:':<25} {'BASE':>10} {'SFT':>10} {'GRPO':>10}")
print("-" * 55)
all_actions = sorted(set(
    list(base_results["per_action"].keys()) +
    list(sft_results["per_action"].keys()) +
    list(grpo_results["per_action"].keys())
))
for action in all_actions:
    b = base_results["per_action"].get(action, 0)
    s = sft_results["per_action"].get(action, 0)
    g = grpo_results["per_action"].get(action, 0)
    print(f"  {action:<23} {b:>10.1%} {s:>10.1%} {g:>10.1%}")


  COMPARISON: BASE vs SFT vs GRPO
Metric                          BASE        SFT       GRPO
-------------------------------------------------------
Action accuracy                 0.0%      89.4%      84.0%
Exact match                     0.0%      89.2%      83.2%
Avg reward                    -1.000      0.787      0.675

Per-action accuracy:            BASE        SFT       GRPO
-------------------------------------------------------
  bet                           0.0%      70.0%      65.0%
  call                          0.0%      89.0%      89.8%
  check                         0.0%      93.7%      92.1%
  fold                          0.0%      95.0%      91.7%
  raise                         0.0%      85.1%      63.8%


## 8. Sample predictions (first 20)

In [12]:
print(f"{'Expected':<15} {'SFT Predicted':<15} {'SFT Rew':>8} {'GRPO Predicted':<15} {'GRPO Rew':>8}")
print("-" * 65)
for i in range(min(20, NUM_TEST_EXAMPLES)):
    s = sft_results["results"][i]
    g = grpo_results["results"][i]
    print(f"{s['expected']:<15} {s['predicted']:<15} {s['reward']:>8.1f} {g['predicted']:<15} {g['reward']:>8.1f}")

Expected        SFT Predicted    SFT Rew GRPO Predicted  GRPO Rew
-----------------------------------------------------------------
call            call                 1.0 call                 1.0
fold            fold                 1.0 fold                 1.0
bet 24          check               -1.0 bet 24               1.0
fold            fold                 1.0 fold                 1.0
check           check                1.0 check                1.0
call            call                 1.0 call                 1.0
raise 41        raise 41             1.0 call                -1.0
call            call                 1.0 call                 1.0
fold            fold                 1.0 fold                 1.0
fold            fold                 1.0 fold                 1.0
raise 89        raise 89             1.0 raise 89             1.0
fold            fold                 1.0 fold                 1.0
check           check                1.0 check                1.0
check     

## 9. Save results to CSV

In [13]:
import csv

output_path = "/kaggle/working/evaluation_results.csv"
with open(output_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["expected", "sft_predicted", "sft_reward", "grpo_predicted", "grpo_reward"])
    for s, g in zip(sft_results["results"], grpo_results["results"]):
        writer.writerow([s["expected"], s["predicted"], s["reward"], g["predicted"], g["reward"]])

print(f"Results saved to {output_path}")

Results saved to /kaggle/working/evaluation_results.csv
